In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
data <- data[!is.na(data$Atyp_10pct_Z), ]
dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2814103      78

[1] 2760385      78

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2760385      78

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total)

NOTES: 4 observations removed because of NA values (RHS: 4).
       17,669/0 fixed-effects (78,042 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,682,339
Fixed-effects: author_id: 61,171,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error   z value
fac_pubBSF                                    0.084090   0.006064  13.86751
num_author_log                                0.257008   0.004598  55.89647
num_inst_log                                  0.009570   0.005837   1.63956
num_country_log                              -0.281855   0.008629 -32.66287
num_reference_log                             0.154881   0.002965  52.23039
leadership_global_classallNorth               0.133238   0.008757  15.21431
leadership_global_classallSouth              -0.014809   0.013789  -1.07397
mean_career_age_log                           0.031533   0.004870   6.47492
inst_h_index_log                             -0.024496   0.003712  -6.59848
source_h_index_log                           -0.044639   0.002189 -20.39661
core_sourceCore    

In [13]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines_vars <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [14]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("fac_pub", paper_vars, journal_vars),
                           se = "iid",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & novelbin\\  
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & 0.084$^{***}$\\   
                                       & (0.006)\\   
   num\_author\_log                    & 0.257$^{***}$\\   
                                       & (0.005)\\   
   num\_inst\_log                      & 0.010\\   
                                       & (0.006)\\   
   num\_country\_log                   & -0.282$^{***}$\\   
                                       & (0.009)\\   
   num\_reference\_log                 & 0.155$^{***}$\\   
                                       & (0.003)\\   
   leadership\_global\_classallNorth   & 0.133$^{***}$\\   
                                       & (0.009)\\   
   leadership\_global\_classallSouth   & -0.015\\   
                                       & (0.014)\\   

In [15]:
# 每组 reg_class 的平均预测概率
margins_eff_reg_class <- avg_predictions(model_total, variables = "fac_pub")
margins_eff_reg_class

fac_pub,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonBSF,0.3640975,0.005828193,62.47176,0,Inf,0.3526744,0.3755205
BSF,0.3792797,0.005912440,64.14944,0,Inf,0.3676915,0.3908679


In [16]:
fname = paste0(main_path, "UseBSForNot/predict_ns_1211.csv")
write.csv(margins_eff_reg_class, fname, row.names = FALSE)

In [17]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", disciplines, " | author_id + year")
)

model_total1 <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total1)

NOTE: 17,669/0 fixed-effects (78,042 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,682,343
Fixed-effects: author_id: 61,171,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error   z value
fac_pubBSF                                    0.134586   0.005920  22.73414
Arts.and.Humanities                           0.547507   0.028279  19.36101
Biochemistry..Genetics.and.Molecular.Biology  0.508124   0.005482  92.68667
Business..Management.and.Accounting           0.048578   0.036088   1.34608
Chemical.Engineering                          0.058646   0.007811   7.50850
Chemistry                                     0.078483   0.004832  16.24242
Computer.Science                              0.389644   0.011231  34.69225
Decision.Sciences                             0.435449   0.030103  14.46508
Dentistry                                     0.697746   0.030802  22.65268
Earth.and.Planetary.Sciences                  0.206512   0.007729  26.72079
Economics..Economet

In [18]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", disciplines, " | author_id + year")
)

model_total2 <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total2)

NOTES: 4 observations removed because of NA values (RHS: 4).
       17,669/0 fixed-effects (78,042 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,682,339
Fixed-effects: author_id: 61,171,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error    z value
fac_pubBSF                                    0.078153   0.006056  12.905110
num_author_log                                0.247552   0.004571  54.159598
num_inst_log                                  0.008504   0.005836   1.457308
num_country_log                              -0.281040   0.008628 -32.571792
num_reference_log                             0.146328   0.002927  49.997854
leadership_global_classallNorth               0.130515   0.008756  14.905843
leadership_global_classallSouth              -0.012785   0.013789  -0.927165
mean_career_age_log                           0.031097   0.004870   6.385825
inst_h_index_log                             -0.027798   0.003706  -7.500455
Arts.and.Humanities                           0.580509   0.028363  20.467246
Biochemi

In [19]:
fml <- as.formula(
  paste0("novelbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total3 <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total3)

NOTES: 4 observations removed because of NA values (RHS: 4).
       17,669/0 fixed-effects (78,042 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novelbin
Observations: 2,682,339
Fixed-effects: author_id: 61,171,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error   z value
fac_pubBSF                                    0.084090   0.006064  13.86751
num_author_log                                0.257008   0.004598  55.89647
num_inst_log                                  0.009570   0.005837   1.63956
num_country_log                              -0.281855   0.008629 -32.66287
num_reference_log                             0.154881   0.002965  52.23039
leadership_global_classallNorth               0.133238   0.008757  15.21431
leadership_global_classallSouth              -0.014809   0.013789  -1.07397
mean_career_age_log                           0.031533   0.004870   6.47492
inst_h_index_log                             -0.024496   0.003712  -6.59848
source_h_index_log                           -0.044639   0.002189 -20.39661
core_sourceCore    

In [20]:
model1 <- model_total1
model2 <- model_total2
model3 <- model_total3

country_level_reg <- etable(
    list(model1, model2, model3),
    keep = c("fac_pub", paper_vars, journal_vars, disciplines_vars ),
    se = "iid",
    # cluster = ~ paper_year + paper_field,
    tex = TRUE,
    digits = 3
)
# writeLines(country_level_reg, con = "selfpromt_reg_1014.tex")

In [21]:
country_level_reg

\begingroup
\centering
\begin{tabular}{lccc}
   \tabularnewline \midrule \midrule
   Dependent Variable: & \multicolumn{3}{c}{novelbin}\\
   Model:                                       & (1)            & (2)            & (3)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                                  & 0.135$^{***}$  & 0.078$^{***}$  & 0.084$^{***}$\\   
                                                & (0.006)        & (0.006)        & (0.006)\\   
   Arts.and.Humanities                          & 0.548$^{***}$  & 0.581$^{***}$  & 0.575$^{***}$\\   
                                                & (0.028)        & (0.028)        & (0.028)\\   
   Biochemistry..Genetics.and.Molecular.Biology & 0.508$^{***}$  & 0.499$^{***}$  & 0.499$^{***}$\\   
                                                & (0.005)        & (0.005)        & (0.005)\\   
   Business..Management.and.Accounting          & 0.049          & 0.069$^{*}$    & 0.063$^{*}$\\   
                                   